# Notebook 3 — Train Outfit Compatibility MLP

Trains a pairwise compatibility MLP on FashionCLIP embeddings of the
Cleaned Maryland Polyvore dataset.

**Input:** `outfit_ids.npy`, `top_embeddings.npy`, `bottom_embeddings.npy` (from Notebook 2)

**Output:** `compatibility_mlp.onnx` and `compatibility_mlp_int8.onnx`

**Pair construction:**
- Positive (label=1): top[i] + bottom[i] from the same outfit
- Negative (label=0): top[i] + bottom[j] where i ≠ j (random cross-outfit pairing)
- 12,505 positive + 12,505 negative = 25,010 total pairs

**MLP input (per pair):**
`[emb_top, emb_bottom, emb_top − emb_bottom, emb_top ⊙ emb_bottom]` → 2048-dim

**Target metric:** AUC ≥ 0.83 on held-out test pairs

**Future expansion note:** When more garment categories (shoes, outerwear, accessories)
are added to EfficientNetB3, extend this notebook to include cross-category pairs
from a richer dataset and add a category-pair one-hot to the MLP input.

**Run on:** Kaggle GPU or Colab T4.

In [ ]:
# ── 1. Install dependencies ──────────────────────────────────────────────────
!pip install -q torch onnx onnxruntime scikit-learn matplotlib numpy tqdm

In [ ]:
# ── 2. Configure paths ────────────────────────────────────────────────────────
# Kaggle: point to the output dataset from embed_polyvore.ipynb
# Colab:  upload the .npy files to Drive and update paths

EMBEDDINGS_DIR = "/kaggle/input/polyvore-embeddings"
OUTPUT_DIR     = "/kaggle/working"
SEED           = 42

import os
assert os.path.isdir(EMBEDDINGS_DIR), f"Embeddings not found: {EMBEDDINGS_DIR}"
print("Paths OK")

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from tqdm import tqdm

torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

In [ ]:
# ── 3. Load embeddings ────────────────────────────────────────────────────────
outfit_ids      = np.load(f"{EMBEDDINGS_DIR}/outfit_ids.npy", allow_pickle=True)
top_embs        = np.load(f"{EMBEDDINGS_DIR}/top_embeddings.npy")
bottom_embs     = np.load(f"{EMBEDDINGS_DIR}/bottom_embeddings.npy")

N = len(outfit_ids)
print(f"Outfits: {N:,}")
print(f"top_embs:    {top_embs.shape}")
print(f"bottom_embs: {bottom_embs.shape}")
assert top_embs.shape == bottom_embs.shape == (N, 512)

In [ ]:
# ── 4. Build pair dataset ─────────────────────────────────────────────────────
# Positive pairs: same-outfit top+bottom
pos_top    = top_embs.copy()     # (N, 512)
pos_bottom = bottom_embs.copy()  # (N, 512)
pos_labels = np.ones(N, dtype=np.float32)

# Negative pairs: shuffle bottom index so top[i] is paired with bottom[j], j≠i
rng = np.random.default_rng(SEED)
neg_bottom_idx = rng.permutation(N)
# Ensure no accidental same-outfit match
collision_mask = neg_bottom_idx == np.arange(N)
neg_bottom_idx[collision_mask] = (neg_bottom_idx[collision_mask] + 1) % N

neg_top    = top_embs.copy()
neg_bottom = bottom_embs[neg_bottom_idx]
neg_labels = np.zeros(N, dtype=np.float32)

# Combine
all_tops    = np.vstack([pos_top,    neg_top])    # (2N, 512)
all_bottoms = np.vstack([pos_bottom, neg_bottom]) # (2N, 512)
all_labels  = np.concatenate([pos_labels, neg_labels])  # (2N,)

print(f"Total pairs: {len(all_labels):,}  (positive: {pos_labels.sum():.0f}, negative: {neg_labels.sum():.0f})")

In [ ]:
# ── 5. Build MLP input features ───────────────────────────────────────────────
# Input: [top, bottom, top−bottom, top⊙bottom] → 2048-dim
# This representation captures: individual style + combined style + style distance + shared features

def build_features(tops, bottoms):
    diff    = tops - bottoms
    product = tops * bottoms
    return np.concatenate([tops, bottoms, diff, product], axis=1)  # (N, 2048)

X = build_features(all_tops, all_bottoms).astype(np.float32)
y = all_labels

print(f"Feature matrix: {X.shape}")

In [ ]:
# ── 6. Train / val / test split ───────────────────────────────────────────────
# Stratified to preserve 50/50 ratio in all splits
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.10, random_state=SEED, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.111, random_state=SEED, stratify=y_train_val
)  # 0.111 of 0.9 ≈ 0.10 overall → 80/10/10 split

print(f"Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}")

In [ ]:
# ── 7. Dataset and DataLoader ─────────────────────────────────────────────────
class PairDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y).unsqueeze(1)  # (N, 1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(PairDataset(X_train, y_train), batch_size=512, shuffle=True,  num_workers=2)
val_loader   = DataLoader(PairDataset(X_val,   y_val),   batch_size=512, shuffle=False, num_workers=2)
test_loader  = DataLoader(PairDataset(X_test,  y_test),  batch_size=512, shuffle=False, num_workers=2)

In [ ]:
# ── 8. Define MLP architecture ────────────────────────────────────────────────
class CompatibilityMLP(nn.Module):
    """
    Pairwise outfit compatibility scorer.
    Input:  2048-dim feature vector [top, bottom, top-bottom, top*bottom]
    Output: scalar in (0, 1) — 1 = compatible, 0 = incompatible

    Future expansion: add a category_pair one-hot to input when more
    garment categories (shoes, outerwear) are supported.
    """
    def __init__(self, input_dim=2048, dropout1=0.3, dropout2=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(dropout1),

            nn.Linear(512, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout2),

            nn.Linear(128, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x)

model = CompatibilityMLP().to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")

In [ ]:
# ── 9. Training loop ──────────────────────────────────────────────────────────
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", patience=3, factor=0.5, min_lr=1e-5
)

MAX_EPOCHS    = 30
PATIENCE      = 5    # early stopping patience
best_val_auc  = 0.0
best_state    = None
patience_cnt  = 0

train_losses, val_aucs = [], []

def evaluate(loader):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for X_b, y_b in loader:
            X_b = X_b.to(DEVICE)
            out = model(X_b).cpu().numpy()
            preds.append(out)
            targets.append(y_b.numpy())
    preds   = np.concatenate(preds).flatten()
    targets = np.concatenate(targets).flatten()
    return roc_auc_score(targets, preds)

print("Training ...")
for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(X_b), y_b)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(X_b)

    epoch_loss /= len(train_loader.dataset)
    val_auc = evaluate(val_loader)
    scheduler.step(val_auc)

    train_losses.append(epoch_loss)
    val_aucs.append(val_auc)

    print(f"Epoch {epoch:02d}/{MAX_EPOCHS}  loss={epoch_loss:.4f}  val_auc={val_auc:.4f}  lr={optimizer.param_groups[0]['lr']:.2e}")

    if val_auc > best_val_auc:
        best_val_auc = val_auc
        best_state   = {k: v.clone() for k, v in model.state_dict().items()}
        patience_cnt = 0
    else:
        patience_cnt += 1
        if patience_cnt >= PATIENCE:
            print(f"Early stopping at epoch {epoch} (best val AUC: {best_val_auc:.4f})")
            break

model.load_state_dict(best_state)
print(f"\nBest val AUC: {best_val_auc:.4f}")

In [ ]:
# ── 10. Plot training curves ──────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(train_losses)
ax1.set_title("Training Loss (BCE)")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.grid(True)

ax2.plot(val_aucs)
ax2.axhline(y=0.83, color="r", linestyle="--", label="Target AUC 0.83")
ax2.set_title("Validation AUC")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("AUC")
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/training_curves.png", dpi=120)
plt.show()

In [ ]:
# ── 11. Evaluate on test set ──────────────────────────────────────────────────
model.eval()
test_preds, test_targets = [], []
with torch.no_grad():
    for X_b, y_b in test_loader:
        out = model(X_b.to(DEVICE)).cpu().numpy()
        test_preds.append(out)
        test_targets.append(y_b.numpy())

test_preds   = np.concatenate(test_preds).flatten()
test_targets = np.concatenate(test_targets).flatten()

test_auc = roc_auc_score(test_targets, test_preds)
test_acc = accuracy_score(test_targets, (test_preds >= 0.5).astype(int))

print(f"Test AUC:      {test_auc:.4f}  (target ≥ 0.83)")
print(f"Test Accuracy: {test_acc:.4f}  (at 0.5 threshold)")

# Score distribution
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(test_preds[test_targets == 1], bins=50, alpha=0.6, label="Compatible (positive)")
ax.hist(test_preds[test_targets == 0], bins=50, alpha=0.6, label="Incompatible (negative)")
ax.axvline(x=0.5, color="k", linestyle="--", label="Threshold 0.5")
ax.axvline(x=0.7, color="r", linestyle="--", label="Buy verdict threshold 0.7")
ax.set_title("Score Distribution on Test Set")
ax.set_xlabel("Compatibility Score")
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/score_distribution.png", dpi=120)
plt.show()

In [ ]:
# ── 12. Export to ONNX ────────────────────────────────────────────────────────
import onnx

ONNX_PATH = f"{OUTPUT_DIR}/compatibility_mlp.onnx"

model.eval().cpu()
dummy_input = torch.randn(1, 2048)

torch.onnx.export(
    model,
    dummy_input,
    ONNX_PATH,
    input_names=["features"],
    output_names=["score"],
    dynamic_axes={
        "features": {0: "batch_size"},
        "score":    {0: "batch_size"},
    },
    opset_version=17,
    do_constant_folding=True,
)

onnx.checker.check_model(ONNX_PATH)
print(f"ONNX model saved: {ONNX_PATH} ({os.path.getsize(ONNX_PATH) / 1e6:.1f} MB)")

In [ ]:
# ── 13. INT8 quantization ─────────────────────────────────────────────────────
from onnxruntime.quantization import quantize_dynamic, QuantType

INT8_PATH = f"{OUTPUT_DIR}/compatibility_mlp_int8.onnx"
quantize_dynamic(ONNX_PATH, INT8_PATH, weight_type=QuantType.QInt8)
print(f"INT8 model saved: {INT8_PATH} ({os.path.getsize(INT8_PATH) / 1e6:.1f} MB)")

In [ ]:
# ── 14. Benchmark inference speed on CPU ──────────────────────────────────────
import onnxruntime as ort
import time

def benchmark_mlp(model_path, batch_size=50, n_runs=200):
    """Simulates scoring 50 wardrobe items against one new item."""
    sess = ort.InferenceSession(model_path, providers=["CPUExecutionProvider"])
    inp  = np.random.randn(batch_size, 2048).astype(np.float32)
    for _ in range(10):  # warmup
        sess.run(["score"], {"features": inp})
    t0 = time.perf_counter()
    for _ in range(n_runs):
        sess.run(["score"], {"features": inp})
    return (time.perf_counter() - t0) / n_runs * 1000

fp32_ms = benchmark_mlp(ONNX_PATH)
int8_ms = benchmark_mlp(INT8_PATH)

print(f"FP32 ONNX (50 items): {fp32_ms:.2f} ms")
print(f"INT8 ONNX (50 items): {int8_ms:.2f} ms")
print(f"Speedup: {fp32_ms / int8_ms:.2f}x")

In [ ]:
# ── 15. Validate with a realistic example ─────────────────────────────────────
# Re-embed a few actual Polyvore outfit pairs and check the scores make sense.
# Same outfit → score should be high. Mismatched → score should be low.

sess_int8 = ort.InferenceSession(INT8_PATH, providers=["CPUExecutionProvider"])

def score_pair(top_emb, bottom_emb):
    t = top_emb.reshape(1, -1).astype(np.float32)
    b = bottom_emb.reshape(1, -1).astype(np.float32)
    features = np.concatenate([t, b, t - b, t * b], axis=1)
    return float(sess_int8.run(["score"], {"features": features})[0][0][0])

print("Sample compatibility scores:")
print("(These should be higher for same-outfit pairs than cross-outfit pairs)\n")

for i in range(5):
    same  = score_pair(top_embs[i], bottom_embs[i])
    cross = score_pair(top_embs[i], bottom_embs[(i + 1000) % N])
    print(f"Outfit {outfit_ids[i]:>12s}  same={same:.3f}  cross-outfit={cross:.3f}")

In [ ]:
# ── 16. Download model files ──────────────────────────────────────────────────
# Place both files in backend/model/ in the app repo.
try:
    from google.colab import files
    files.download(ONNX_PATH)
    files.download(INT8_PATH)
    files.download(f"{OUTPUT_DIR}/training_curves.png")
    files.download(f"{OUTPUT_DIR}/score_distribution.png")
except ImportError:
    print("Kaggle: download from the output panel on the right.")
    for f in [ONNX_PATH, INT8_PATH]:
        print(f"  {f} ({os.path.getsize(f) / 1e6:.1f} MB)")

print("\nNext step: place compatibility_mlp_int8.onnx in backend/model/")